In [ ]:
# | default_exp features.endmotif_genomewide

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator, parse_array

log = structlog.get_logger()

In [ ]:
# | export


class EndMotifGenomewideEvaluator(FeatureEvaluator):
    """Extracts 4-mer fragment end motif frequencies for genomewide regions."""

    name = "EndMotifGenomewide"
    source_file = ".EndMotif.parquet"
    tier = 3
    category = "motifs"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            if "Motif" in cols and "Frequency" in cols:
                for row in df.to_dict("records"):
                    m = str(row["Motif"])
                    if pd.notna(row["Frequency"]):
                        extracted[f"endmotif_gw_{m}"] = float(row["Frequency"])

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
evaluator = EndMotifGenomewideEvaluator()
df = pd.DataFrame([{"Motif": "AAAA", "Frequency": 0.05}])
res = evaluator.extract(df)
assert res["endmotif_gw_AAAA"] == 0.05